# Stuttering Detection: K-Nearest Neighbors (KNN) Analysis
**Course**: CS204T (Artificial Intelligence)  
**Team**: 18  
**Focus**: Distance-Based Classification on WavLM Manifolds

---

## Step 1: Initialization & Environment
We use the project's standardized modular engine to load high-dimensional WavLM embeddings. KNN is a 'Memory-Based' learner that classifies stutters by looking at the most similar audio samples in our training database.

In [ ]:
import os
import shutil
import numpy as np
from src.extractors import WavLMExtractor
from src.data import DataManager
from src.models import KNNModel

# Paths to our distributed dataset
AUDIO_DIR = "Stuttering Events in Podcasts Dataset/clips/stuttering-clips/clips"
CSV_PATHS = [
    "Stuttering Events in Podcasts Dataset/SEP-28k_labels.csv",
    "Stuttering Events in Podcasts Dataset/fluencybank_labels.csv"
]
FEATURE_DIR = "data/features"
fluent_dir = os.path.join(FEATURE_DIR, "fluent")
disfluent_dir = os.path.join(FEATURE_DIR, "disfluent")

## Step 2 (Optional): Operational Mode for Data Extraction
Toggle how you want to handle your audio data for this session.
* `SKIP_EXTRACTION`: Uses features already on disk (Default).
* `FORCE_EXTRACT`: Analyzes raw audio for new files (Resumable).
* `CLEAN_START`: Wipes the database and re-extracts from zero.

In [ ]:
# Operational Flags
SKIP_EXTRACTION = True
FORCE_EXTRACT = False
CLEAN_START = False
NUM_CLIPS_TO_EXTRACT = 1000

if CLEAN_START:
    if os.path.exists(FEATURE_DIR):
        shutil.rmtree(FEATURE_DIR)
    print("[System] Clean start initiated. Wiped feature database.")

if not SKIP_EXTRACTION or CLEAN_START or FORCE_EXTRACT:
    extractor = WavLMExtractor("microsoft/wavlm-base")
    label_dict = DataManager.generate_label_dict(CSV_PATHS, filter_quality=True)
    
    # Native randomized sampling of the dataset
    extractor.extract_from_dir(
        AUDIO_DIR, 
        output_dir=FEATURE_DIR, 
        label_dict=label_dict, 
        limit=NUM_CLIPS_TO_EXTRACT, 
        random_sample=True
    )
else:
    print("[System] Skipping extraction. Using existing data on disk.")

## Step 3: Data Preparation Engine
We load our distributed feature files, perform randomized splitting, and apply **oversampling**. Standardization is MANDATORY for KNN because features with larger numerical ranges will otherwise dominate the distance calculation.

In [ ]:
label_dict = DataManager.generate_label_dict(CSV_PATHS, filter_quality=True)
manager = DataManager(None, None)

# Loading .npy features
X, y = manager.load_from_folders(fluent_dir, disfluent_dir)
X_train, X_val, X_test, y_train, y_val, y_test = manager.get_splits(test_size=0.15, val_size=0.15)

# Oversampling training data only
X_train_bal, y_train_bal = manager.balance_data(X_train, y_train, strategy="oversample")

# Anti-Leakage Standard Selection (MANDATORY for KNN accuracy)
X_train_final = manager.preprocess(X_train_bal, method="standard", fit=True)
X_test_final = manager.preprocess(X_test, method="standard", fit=False)

## Step 4: Model Execution - K-Nearest Neighbors
We use **K=5** neighbors. The model calculates the 'closeness' of each test sample to known stuttering samples in the 768-dimensional WavLM space.

In [ ]:
knn_model = KNNModel("KNN_K5", k=5)
knn_model.train(X_train_final, y_train_bal)

print("\n--- Evaluation on Unseen Test Set ---")
knn_model.evaluate(X_test_final, y_test)

## Step 5: Research Conclusion - The Curse of Dimensionality
While KNN is intuitive, it often struggles with the 'Curse of Dimensionality' in 768-dimensional spaces. 

**Observation**: If Accuracy is high but Recall is low, it suggests that stuttering clusters are not purely spherical in this high-dimensional manifold, prompting the need for the more complex neural networks explored in our other modules.